In [1]:
import os
import numpy as np
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from sklearn.metrics import precision_recall_curve, auc
import logging

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from mamba_ssm import Mamba

# ==========================================
# Global config & logging
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available!")

device = torch.device("cuda")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

TARGET_LEN = 6000
INPUT_CHANNEL = 3
OUTPUT_CHANNEL = 3
FS = 100.0
PICK_TOLERANCE = 50

PICK_THRESHOLD_P = 0.02
PICK_THRESHOLD_S = 0.08

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 5
LEARNING_RATE = 1e-4

NOISE_BAND = [1, 45]
RANDOM_SNR_RANGE = [0, 20]

OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REAL_NOISE_PATH = "./stead_real_noise_50000.npy"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_DIR, 'training.log'), mode='w'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info(f"✅ Using GPU: {torch.cuda.get_device_name(device)}")
logger.info(f"✅ Output directory: {OUTPUT_DIR}")
logger.info(f"✅ Real noise path: {REAL_NOISE_PATH}")
logger.info(f"✅ Loss: Categorical Cross Entropy")
logger.info(f"✅ Preprocessing: no bandpass filter on input waveforms (only synthetic noise retains bandpass)")

# ==========================================
# Data preprocessing & augmentation
# ==========================================
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=-1)
    return y

def generate_bandpass_noise(length, fs, band, order=4):
    lowcut, highcut = band
    white_noise = np.random.randn(length)
    band_noise = bandpass_filter(white_noise, lowcut, highcut, fs, order)
    return band_noise

def add_noise_with_snr(signal, snr_db, fs, noise_band):
    signal_noisy = np.zeros_like(signal)
    for ch in range(signal.shape[1]):
        sig = signal[:, ch]
        sig_power = np.mean(sig ** 2)
        noise_power = sig_power / (10 ** (snr_db / 10))
        noise = generate_bandpass_noise(len(sig), fs, noise_band)
        noise_scaled = noise * np.sqrt(noise_power / (np.mean(noise ** 2) + 1e-10))
        signal_noisy[:, ch] = sig + noise_scaled
    return signal_noisy

def shift_waveform_and_label(waveform, label, max_shift=500):
    shift = np.random.randint(-max_shift, max_shift)
    shifted_wave = np.zeros_like(waveform)
    shifted_label = np.zeros_like(label)

    if shift > 0:
        shifted_wave[shift:, :] = waveform[:-shift, :]
        shifted_label[shift:, :] = label[:-shift, :]
    elif shift < 0:
        shifted_wave[:shift, :] = waveform[-shift:, :]
        shifted_label[:shift, :] = label[-shift:, :]
    else:
        shifted_wave = waveform
        shifted_label = label
    return shifted_wave, shifted_label, shift

def normalize_single_sample(x):
    x_normalized = np.zeros_like(x, dtype=np.float32)
    for ch in range(INPUT_CHANNEL):
        mean = np.mean(x[:, ch])
        std = np.std(x[:, ch])
        if std < 1e-6:
            std = 1.0
        x_normalized[:, ch] = (x[:, ch] - mean) / std
    return x_normalized

# ==========================================
# Basic blocks
# ==========================================
class StochasticDepthDropout(nn.Module):
    def __init__(self, drop_rate):
        super().__init__()
        self.drop_rate = drop_rate

    def forward(self, layer_output, residual):
        if self.training:
            keep_prob = 1 - self.drop_rate
            if torch.rand(1).item() < keep_prob:
                return residual + (layer_output / keep_prob)
            else:
                return residual
        else:
            return residual + layer_output

class DilatedConvBlock(nn.Module):
    """Two dilated conv layers with residual connection."""
    def __init__(self, in_channels, filters, kernel_size=13, dilation_rates=[1, 2], dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size,
                               dilation=dilation_rates[0], padding='same')
        self.norm1 = nn.LayerNorm(filters)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(filters, filters, kernel_size,
                               dilation=dilation_rates[1], padding='same')
        self.norm2 = nn.LayerNorm(filters)
        self.act2 = nn.ReLU()

        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv1d(in_channels, filters, 1) if in_channels != filters else nn.Identity()

    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv1(x)
        x = self.norm1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act2(x)
        x = self.dropout(x)
        return x + residual

class DualMambaAttentionBlock(nn.Module):
    """Multi-head attention + bidirectional Mamba with stochastic depth."""
    def __init__(self, channels, num_heads=16, dropout=0.1, drop_rate=0.0):
        super().__init__()
        self.channels = channels

        self.pre_conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=13, padding='same'),
            nn.ReLU()
        )

        self.mha_norm = nn.LayerNorm(channels)
        self.mha = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.sdd_mha = StochasticDepthDropout(drop_rate)

        self.mamba_fwd = Mamba(
            d_model=channels,
            d_state=8,
            d_conv=2,
            expand=2,
        )
        self.mamba_bwd = Mamba(
            d_model=channels,
            d_state=8,
            d_conv=2,
            expand=2,
        )
        self.fusion = nn.Linear(2 * channels, channels)
        self.mamba_norm = nn.LayerNorm(channels)
        self.sdd_mamba = StochasticDepthDropout(drop_rate)

    def forward(self, x):
        residual_att = x

        x = self.pre_conv(x)
        x = x.permute(0, 2, 1)

        x_norm = self.mha_norm(x)
        att_out, _ = self.mha(x_norm, x_norm, x_norm)
        att_out = att_out.permute(0, 2, 1)
        x = self.sdd_mha(att_out, residual_att)

        residual_mamba = x
        temp_x = x.permute(0, 2, 1)

        out_fwd = self.mamba_fwd(temp_x)
        temp_x_rev = torch.flip(temp_x, dims=[1])
        out_bwd = self.mamba_bwd(temp_x_rev)
        out_bwd = torch.flip(out_bwd, dims=[1])

        out_combined = torch.cat([out_fwd, out_bwd], dim=-1)
        temp_x = self.fusion(out_combined)
        temp_x = self.mamba_norm(temp_x)

        temp_x = temp_x.permute(0, 2, 1)
        x = self.sdd_mamba(temp_x, residual_mamba)

        return x

class EncoderBlock(nn.Module):
    """Dilated conv block + downsampling. Returns skip and downsampled features."""
    def __init__(self, in_channels, out_channels, dilation_rates, dropout=0.1):
        super().__init__()
        self.dilated = DilatedConvBlock(in_channels, out_channels, 13, dilation_rates, dropout)
        self.down = nn.Conv1d(out_channels, out_channels, 13, stride=2, padding=6)

    def forward(self, x):
        skip = self.dilated(x)
        down = F.relu(self.down(skip))
        return skip, down

class DecoderBlock(nn.Module):
    """Upsampling + concatenation with skip connection + two conv layers."""
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=13,
                                     stride=2, padding=6, output_padding=1)
        conv_in = out_channels + skip_channels
        self.conv = nn.Sequential(
            nn.Conv1d(conv_in, out_channels, 13, padding=6),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, 13, padding=6),
            nn.ReLU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = F.relu(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

class PhaseNet_Custom_DoubleMamba(nn.Module):
    """U-Net style network with dual Mamba attention bottleneck."""
    def __init__(self, num_heads=16):
        super().__init__()

        self.enc1 = EncoderBlock(INPUT_CHANNEL, 16, [1, 2])
        self.enc2 = EncoderBlock(16, 32, [3, 4])
        self.enc3 = EncoderBlock(32, 64, [5, 6])
        self.enc4 = EncoderBlock(64, 128, [7, 8])

        self.dma1 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.0)
        self.dma2 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.1)

        self.dec4 = DecoderBlock(128, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec2 = DecoderBlock(64, 32, 32)
        self.dec1 = DecoderBlock(32, 16, 16)

        self.final_conv = nn.Conv1d(16, 3, 1, padding='same')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, C, T)

        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        x = self.dma1(x)
        x = self.dma2(x)

        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        x = self.final_conv(x)
        x = x.permute(0, 2, 1)            # (B, T, C)
        x = torch.softmax(x, dim=-1)
        return x

# ==========================================
# Dataset
# ==========================================
class AugmentedDataset(Dataset):
    def __init__(self, X, y, is_aug=True):
        self.X = X
        self.y = y
        self.is_aug = is_aug

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_raw = self.X[idx].copy()
        y = self.y[idx].copy()

        if self.is_aug:
            if np.random.rand() < 0.9:
                x_raw, y, _ = shift_waveform_and_label(x_raw, y, max_shift=500)
            if np.random.rand() < 0.5:
                snr = np.random.uniform(RANDOM_SNR_RANGE[0], RANDOM_SNR_RANGE[1])
                x_raw = add_noise_with_snr(x_raw, snr, FS, NOISE_BAND)
            if np.random.rand() < 0.1:
                num_drop = np.random.choice([1, 2])
                channels_to_drop = np.random.choice(INPUT_CHANNEL, num_drop, replace=False)
                x_raw[:, channels_to_drop] = 0.0

            x_norm = normalize_single_sample(x_raw)
        else:
            x_norm = normalize_single_sample(x_raw)

        return torch.tensor(x_norm, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ==========================================
# Label generation
# ==========================================
def generate_triangle_label(length, peak_idx, width=40):
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)

    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)

    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

# ==========================================
# Dataset loading
# ==========================================
def load_stead_dataset():
    logger.info("="*60)
    logger.info("  Loading STEAD Dataset | real noise only, no synthetic negatives")
    logger.info("="*60)

    X_original = np.load("chunk2_stead_X.npy").astype(np.float32)
    y_2channel = np.load("chunk2_stead_y.npy").astype(np.float32)
    y = generate_3channel_labels(y_2channel, width_p=20, width_s=40)

    indices = np.arange(len(X_original))
    np.random.shuffle(indices)
    X_original = X_original[indices]
    y = y[indices]

    n_total_eq = len(X_original)
    n_train_eq = int(n_total_eq * 0.7)
    n_val_eq = int(n_total_eq * 0.2)

    X_train_eq, y_train_eq = X_original[:n_train_eq], y[:n_train_eq]
    X_val_eq, y_val_eq = X_original[n_train_eq:n_train_eq+n_val_eq], y[n_train_eq:n_train_eq+n_val_eq]
    X_test_eq, y_test_eq = X_original[n_train_eq+n_val_eq:], y[n_train_eq+n_val_eq:]

    logger.info(f"Loading STEAD real noise: {REAL_NOISE_PATH}")
    all_real_noise = np.load(REAL_NOISE_PATH).astype(np.float32)
    total_noise_num = len(all_real_noise)
    logger.info(f"✅ Real noise loaded: {total_noise_num} samples")

    assert total_noise_num >= 50000, "Not enough noise samples!"
    all_real_noise = all_real_noise[:50000]
    total_noise_num = 50000

    n_train_noise = int(total_noise_num * 0.7)
    n_val_noise = int(total_noise_num * 0.2)
    n_test_noise = total_noise_num - n_train_noise - n_val_noise

    noise_indices = np.arange(total_noise_num)
    np.random.shuffle(noise_indices)

    train_noise_indices = noise_indices[:n_train_noise]
    val_noise_indices = noise_indices[n_train_noise:n_train_noise+n_val_noise]
    test_noise_indices = noise_indices[n_train_noise+n_val_noise:]

    X_train_noise = all_real_noise[train_noise_indices]
    y_train_noise = np.zeros((n_train_noise, TARGET_LEN, 3), dtype=np.float32)
    y_train_noise[..., 2] = 1.0

    X_val_noise = all_real_noise[val_noise_indices]
    y_val_noise = np.zeros((n_val_noise, TARGET_LEN, 3), dtype=np.float32)
    y_val_noise[..., 2] = 1.0

    X_test_noise = all_real_noise[test_noise_indices]
    y_test_noise = np.zeros((n_test_noise, TARGET_LEN, 3), dtype=np.float32)
    y_test_noise[..., 2] = 1.0

    X_train = np.concatenate([X_train_eq, X_train_noise], axis=0)
    y_train = np.concatenate([y_train_eq, y_train_noise], axis=0)
    final_train_indices = np.arange(len(X_train))
    np.random.shuffle(final_train_indices)
    X_train = X_train[final_train_indices]
    y_train = y_train[final_train_indices]

    X_val = np.concatenate([X_val_eq, X_val_noise], axis=0)
    y_val = np.concatenate([y_val_eq, y_val_noise], axis=0)
    final_val_indices = np.arange(len(X_val))
    np.random.shuffle(final_val_indices)
    X_val = X_val[final_val_indices]
    y_val = y_val[final_val_indices]

    X_test = np.concatenate([X_test_eq, X_test_noise], axis=0)
    y_test = np.concatenate([y_test_eq, y_test_noise], axis=0)

    logger.info(f"\nFinal dataset statistics:")
    logger.info(f"  ├─ Train: events={len(X_train_eq)}, noise={len(X_train_noise)}, total={len(X_train)}")
    logger.info(f"  ├─ Val:   events={len(X_val_eq)}, noise={len(X_val_noise)}, total={len(X_val)}")
    logger.info(f"  └─ Test:  events={len(X_test_eq)}, noise={len(X_test_noise)}, total={len(X_test)}")
    logger.info("="*60)

    return X_train, y_train, X_val, y_val, X_test, y_test

# ==========================================
# Metrics
# ==========================================
def calc_metrics(y_true, y_pred):
    earthquake_mask = np.max(y_true[..., 0], axis=1) > 0.5
    noise_mask = ~earthquake_mask
    y_true_eq = y_true[earthquake_mask]
    y_pred_eq = y_pred[earthquake_mask]
    y_true_noise = y_true[noise_mask]
    y_pred_noise = y_pred[noise_mask]
    n_eq = len(y_true_eq)
    n_noise = len(y_true_noise)
    metrics = {}

    p_errors_list = []
    s_errors_list = []

    for ch, name in enumerate(['P', 'S']):
        true_idx = np.argmax(y_true_eq[..., ch], axis=1)
        pred_max_prob = np.max(y_pred_eq[..., ch], axis=1)
        pred_idx = np.argmax(y_pred_eq[..., ch], axis=1)

        if name == 'P':
            valid_detect = pred_max_prob >= PICK_THRESHOLD_P
        else:
            valid_detect = pred_max_prob >= PICK_THRESHOLD_S

        valid_idx = np.where(valid_detect)[0]
        sample_errors = (pred_idx[valid_idx] - true_idx[valid_idx]) / FS
        if name == 'P':
            p_errors_list = sample_errors
        else:
            s_errors_list = sample_errors

        pick_error = np.abs(true_idx - pred_idx)
        correct = valid_detect & (pick_error <= PICK_TOLERANCE)
        tp = np.sum(correct)
        fn = n_eq - tp
        mae = np.mean(pick_error[correct]) if tp > 0 else np.inf

        noise_max_prob = np.max(y_pred_noise[..., ch], axis=1)
        if name == 'P':
            fp = np.sum(noise_max_prob >= PICK_THRESHOLD_P)
        else:
            fp = np.sum(noise_max_prob >= PICK_THRESHOLD_S)
        tn = n_noise - fp

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

        y_true_binary = np.concatenate([np.ones(n_eq), np.zeros(n_noise)])
        y_pred_score = np.concatenate([np.max(y_pred_eq[..., ch], axis=1), np.max(y_pred_noise[..., ch], axis=1)])
        prec_curve, rec_curve, _ = precision_recall_curve(y_true_binary, y_pred_score)
        auc_score = auc(rec_curve, prec_curve)

        metrics[f'mae_{name}'] = mae
        metrics[f'tp_{name}'] = tp
        metrics[f'fn_{name}'] = fn
        metrics[f'fp_{name}'] = fp
        metrics[f'tn_{name}'] = tn
        metrics[f'prec_{name}'] = precision
        metrics[f'rec_{name}'] = recall
        metrics[f'f1_{name}'] = f1
        metrics[f'auc_{name}'] = auc_score

    metrics['n_earthquake'] = n_eq
    metrics['n_noise'] = n_noise
    return metrics, np.array(p_errors_list), np.array(s_errors_list)

# ==========================================
# Training
# ==========================================
def main():
    logger.info("="*80)
    logger.info(f"Custom Double Mamba Model | Categorical Cross Entropy | kernel=13 | heads=16")
    logger.info("="*80)

    X_train, y_train, X_val, y_val, X_test, y_test = load_stead_dataset()

    train_dataset = AugmentedDataset(X_train, y_train, is_aug=True)
    val_dataset = AugmentedDataset(X_val, y_val, is_aug=False)
    test_dataset = AugmentedDataset(X_test, y_test, is_aug=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    model = PhaseNet_Custom_DoubleMamba(num_heads=16).to(device)
    logger.info(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")

    def categorical_cross_entropy(y_pred, y_true):
        y_pred_clamped = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        loss = -torch.mean(torch.sum(y_true * torch.log(y_pred_clamped), dim=-1))
        return loss

    loss_fn = categorical_cross_entropy

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

    best_val_loss = float('inf')
    epochs_no_improve = 0

    logger.info("\n===== Start Training =====")
    for epoch in range(EPOCHS):
        model.train()
        train_loss_sum = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss_sum += loss.item() * X_batch.size(0)
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        all_val_true, all_val_pred = [], []
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
            for X_batch, y_batch in pbar:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                val_loss_sum += loss.item() * X_batch.size(0)
                all_val_true.append(y_batch.cpu().numpy())
                all_val_pred.append(y_pred.cpu().numpy())
        val_loss = val_loss_sum / len(val_loader.dataset)
        scheduler.step(val_loss)

        all_val_true = np.concatenate(all_val_true, axis=0)
        all_val_pred = np.concatenate(all_val_pred, axis=0)
        m, _, _ = calc_metrics(all_val_true, all_val_pred)

        logger.info(f"\n  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        logger.info(f"  P-wave: MAE={m['mae_P']/FS:.3f}s, AUC={m['auc_P']:.4f}, Precision={m['prec_P']:.4f}, Recall={m['rec_P']:.4f}, F1={m['f1_P']:.4f}")
        logger.info(f"  S-wave: MAE={m['mae_S']/FS:.3f}s, AUC={m['auc_S']:.4f}, Precision={m['prec_S']:.4f}, Recall={m['rec_S']:.4f}, F1={m['f1_S']:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_best.pt"))
            logger.info(f"  ✅ Best model saved")
        else:
            epochs_no_improve += 1
            logger.info(f"  ⏳ No improvement: {epochs_no_improve}/{PATIENCE}")
            if epochs_no_improve >= PATIENCE:
                logger.info(f"\nEarly stop triggered!")
                break

    # Test evaluation
    logger.info("\n" + "="*60)
    logger.info("  Test Set Evaluation")
    logger.info("="*60)
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "model_best.pt"), weights_only=True))
    model.eval()

    all_test_true, all_test_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in tqdm(test_loader, desc="Testing"):
            y_pred = model(X_batch.to(device))
            all_test_true.append(y_batch.numpy())
            all_test_pred.append(y_pred.cpu().numpy())

    all_test_true = np.concatenate(all_test_true, axis=0)
    all_test_pred = np.concatenate(all_test_pred, axis=0)
    m, p_errors, s_errors = calc_metrics(all_test_true, all_test_pred)

    logger.info("\n" + "="*80)
    logger.info(f"  Final Test Results (Categorical CE, Kernel=13, Heads=16)")
    logger.info("="*80)
    logger.info(f"  {'Metric':<12} | {'P-wave':<12} | {'S-wave':<12}")
    logger.info("-"*80)
    logger.info(f"  {'MAE(sec)':<12} | {m['mae_P']/FS:<12.3f} | {m['mae_S']/FS:<12.3f}")
    logger.info(f"  {'PR-AUC':<12} | {m['auc_P']:<12.4f} | {m['auc_S']:<12.4f}")
    logger.info(f"  {'TP':<12} | {m['tp_P']:<12.0f} | {m['tp_S']:<12.0f}")
    logger.info(f"  {'TN':<12} | {m['tn_P']:<12.0f} | {m['tn_S']:<12.0f}")
    logger.info(f"  {'FN':<12} | {m['fn_P']:<12.0f} | {m['fn_S']:<12.0f}")
    logger.info(f"  {'FP':<12} | {m['fp_P']:<12.0f} | {m['fp_S']:<12.0f}")
    logger.info(f"  {'Precision':<12} | {m['prec_P']:<12.4f} | {m['prec_S']:<12.4f}")
    logger.info(f"  {'Recall':<12} | {m['rec_P']:<12.4f} | {m['rec_S']:<12.4f}")
    logger.info(f"  {'F1-score':<12} | {m['f1_P']:<12.4f} | {m['f1_S']:<12.4f}")
    logger.info("="*80)

    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_final.pt"))
    logger.info("\n✅ Training finished! Model saved to outputs folder")

if __name__ == "__main__":
    main()

/home/fc/anaconda3/envs/jhwl/lib/python3.11/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
2026-05-29 17:57:36,300 - ✅ Using GPU: NVIDIA GeForce RTX 4090
2026-05-29 17:57:36,301 - ✅ Output directory: ./outputs
2026-05-29 17:57:36,302 - ✅ Real noise path: ./stead_real_noise_50000.npy
2026-05-29 17:57:36,303 - ✅ Loss: Categorical Cross Entropy
2026-05-29 17:57:36,304 - ✅ Preprocessing: no bandpass filter on input waveforms (only synthetic noise retains bandpass)
2026-05-29 17:57:36,309 - ================================================================================
2026-05-29 17:57:36,309 - Custom Double Mamba Model | Categorical Cross Entropy | kernel=13 | heads=16
2026-05-29 17:57:36,312 - ================================================================================
2026-05-29 17:57:36,312 - ============================================================
2026-05-29 17:57:36,313 -   Loading STEAD Dataset | real noise 